# #22 · Manual 2×2 DiD — ATT using U.S. and Canada

**Owner:** @gaurvsingh095  
**Issue:** [#22](https://github.com/Break-Through-Tech/Estee-Lauder-1B-measuring-customer-delight/issues/22)  
**Milestone:** #3 — Simple DiD: control selection & manual 2×2

## Goal

Calculate the manual 2×2 Difference-in-Differences estimate of the **Average Treatment Effect on the Treated (ATT)** using:

- Treated market: **United States**
- Control market: **Canada**
- Outcome: `revenue_per_session`

Formula:

```text
ATT = (US_post − US_pre) − (Canada_post − Canada_pre)
```

Canada is used here because the EDA/control-selection work identified it as the strongest first-pass control market. This notebook does **not** redo control selection; it uses the selected control to produce the treatment-effect estimate.

In [1]:
import sys
sys.path.append("../../src")

import pandas as pd

import simple_did as sd

df = sd.load_panel()
launch = sd.launch_week(df)
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Launch week: {launch.date()}")
df.head()

Loaded 728 rows × 16 columns
Launch week: 2026-04-06


,market,week_index,week_start,sessions,fragrance_orders,fragrance_revenue,conversion_rate,average_order_value,revenue_per_session,paid_media_index,promo_intensity,new_visitor_share,pilot_market,post_launch,digital_feature_available,weeks_from_launch
0,Australia,0,2025-01-06,69314,1870,147050.31,0.026979,78.64,2.1215,96.81,0.2081,0.3308,0,0,0,-65
1,Australia,1,2025-01-13,71010,2011,159108.26,0.028320,79.12,2.2406,100.78,0.2142,0.3870,0,0,0,-64
2,Australia,2,2025-01-20,71268,1954,150249.86,0.027418,76.89,2.1082,89.21,0.2005,0.3705,0,0,0,-63
3,Australia,3,2025-01-27,67993,1632,128219.61,0.024002,78.57,1.8858,104.77,0.2123,0.3519,0,0,0,-62
4,Australia,4,2025-02-03,68280,1952,161268.21,0.028588,82.62,2.3619,101.49,0.1982,0.3558,0,0,0,-61


## Step 1 — Compute the four 2×2 group means

The four cells are:

1. U.S. pre-launch average RPS
2. U.S. post-launch average RPS
3. Canada pre-launch average RPS
4. Canada post-launch average RPS

Showing all four cells makes the arithmetic auditable.

In [2]:
did = sd.manual_did(df, control="Canada")

did_table = pd.DataFrame([
    {"market": did["treated"], "group": "treated", "period": "pre_launch", "avg_revenue_per_session": did["treated_pre"]},
    {"market": did["treated"], "group": "treated", "period": "post_launch", "avg_revenue_per_session": did["treated_post"]},
    {"market": did["control"], "group": "control", "period": "pre_launch", "avg_revenue_per_session": did["control_pre"]},
    {"market": did["control"], "group": "control", "period": "post_launch", "avg_revenue_per_session": did["control_post"]},
])

did_table.round(4)

,market,group,period,avg_revenue_per_session
0,United States,treated,pre_launch,2.6809
1,United States,treated,post_launch,3.0230
2,Canada,control,pre_launch,2.4461
3,Canada,control,post_launch,2.5423


## Step 2 — Calculate the manual DiD estimate

The U.S. changed after launch, but Canada also changed. DiD subtracts Canada’s change from the U.S. change.

In [3]:
did_summary = pd.DataFrame([{
    "method": "manual_2x2_did",
    "treated_market": did["treated"],
    "control_market": did["control"],
    "metric": did["metric"],
    "treated_pre": did["treated_pre"],
    "treated_post": did["treated_post"],
    "treated_change": did["treated_change"],
    "control_pre": did["control_pre"],
    "control_post": did["control_post"],
    "control_change": did["control_change"],
    "att_estimate_rps": did["estimate"],
}])

did_summary.round(4)

,method,treated_market,control_market,metric,treated_pre,treated_post,treated_change,control_pre,control_post,control_change,att_estimate_rps
0,manual_2x2_did,United States,Canada,revenue_per_session,2.6809,3.023,0.3421,2.4461,2.5423,0.0962,0.2459


## Interpretation

The manual DiD estimate is the U.S. before/after change minus Canada’s before/after change.

This answers the milestone’s counterfactual question more directly than the naive estimate:

> How much higher was U.S. `revenue_per_session` after launch compared with what it might have been without the digital feature, using Canada as the counterfactual?

## Assumptions and limitations

For this manual DiD estimate to be interpreted causally, the following assumptions need to be plausible:

1. **Parallel trends:** without the feature, U.S. and Canada would have continued moving similarly after launch. This is the main assumption and should be tested in the pre-launch regression work.
2. **No spillovers:** Canada did not receive the feature and was not affected by the U.S. rollout.
3. **No major U.S.-only shock at launch:** no other U.S.-specific event should explain the extra U.S. lift. This is vulnerable because prior EDA showed paid media, promo intensity, and new visitor share moved around launch.
4. **No anticipation effect:** users/marketing should not have changed behavior before launch in a way that already reflects the feature. This is also vulnerable because paid media started climbing near launch.

So this estimate is a better preliminary business estimate than naive before/after, but it is still not the final causal answer.

In [4]:
registry = sd.save_estimate(
    method="manual_2x2_did",
    treated=did["treated"],
    control=did["control"],
    metric=did["metric"],
    window="all_pre_vs_post",
    estimate=did["estimate"],
    issue=22,
    notes=(
        "Manual 2x2 DiD ATT using Canada as control. More appropriate than naive before/after "
        "because it subtracts Canada's baseline movement. Still preliminary: causal interpretation "
        "requires plausible parallel trends, no spillovers, no anticipation, and no coincident U.S.-only shocks."
    ),
)

registry.tail()

,method,treated,control,metric,window,estimate,ci_low,ci_high,issue,author,recorded_at,notes
0,naive_before_after,United States,,revenue_per_session,all_pre_vs_post,0.342090,NaN,NaN,21,@gaurvsingh095,2026-09-22T18:50:58Z,Naive U.S. before/after change. Descriptive ba...
1,manual_2x2_did,United States,Canada,revenue_per_session,all_pre_vs_post,0.245914,,,22,@gaurvsingh095,2026-09-22T18:51:02Z,Manual 2x2 DiD ATT using Canada as control. Mo...


## Final answer for issue #22

Using Canada as the selected control market, the manual 2×2 DiD estimate is the U.S. change minus Canada’s change. This estimate should be used instead of the naive estimate for the first-stage business readout, with the limitations above clearly stated.